# Task 4 - GAN and Conditional GAN on PlantVillage

This notebook implements a clean, modular pipeline for:

1. Dataset exploration and preprocessing for **Apple** or **Tomato** leaves.
2. Training a standard **GAN** with learning-rate ratio tuning.
3. Saving generated images over time and comparing **Epoch 1 vs Epoch 50**.
4. Upgrading to a **Conditional GAN (cGAN)** with label conditioning (**Healthy** vs **Diseased**).

## Requirements

Install dependencies using one of the following:

```bash
pip install torch torchvision matplotlib pillow numpy tqdm
```

Or create a `requirements.txt` with:

```text
torch>=2.2.0
torchvision>=0.17.0
matplotlib>=3.8.0
pillow>=10.0.0
numpy>=1.26.0
tqdm>=4.66.0
```

In [ ]:
import os
import random
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image


# Reproducibility
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# =============================
# Task 1 - Dataset Exploration
# =============================

class LeafConditionDataset(Dataset):
    """Wrap a subset from ImageFolder and derive binary condition labels.

    condition label:
    - 0 -> healthy
    - 1 -> diseased
    """

    def __init__(self, base_dataset: datasets.ImageFolder, indices: List[int]):
        self.base_dataset = base_dataset
        self.indices = indices

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int):
        original_index = self.indices[idx]
        image, class_idx = self.base_dataset[original_index]
        class_name = self.base_dataset.classes[class_idx].lower()
        condition_label = 0 if "healthy" in class_name else 1
        return image, condition_label


def get_transforms(image_size: int = 64) -> transforms.Compose:
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ])


def build_filtered_dataset(
    dataset_root: str,
    target_plant: str = "Apple",
    image_size: int = 64,
) -> Tuple[datasets.ImageFolder, List[int], LeafConditionDataset]:
    """Load PlantVillage and keep only classes containing target_plant.

    Args:
        dataset_root: Path to PlantVillage root folder for ImageFolder.
        target_plant: Either "Apple" or "Tomato" (case-insensitive).
        image_size: Resize target.
    """

    target_plant = target_plant.lower()
    if target_plant not in {"apple", "tomato"}:
        raise ValueError("target_plant must be either 'Apple' or 'Tomato'.")

    dataset = datasets.ImageFolder(root=dataset_root, transform=get_transforms(image_size))

    keep_indices = []
    for i, (_, class_idx) in enumerate(dataset.samples):
        class_name = dataset.classes[class_idx].lower()
        if target_plant in class_name:
            keep_indices.append(i)

    if not keep_indices:
        raise RuntimeError(
            f"No samples found for plant '{target_plant}'. Check folder names in PlantVillage."
        )

    conditioned_dataset = LeafConditionDataset(dataset, keep_indices)
    return dataset, keep_indices, conditioned_dataset


def make_dataloaders(
    dataset_root: str,
    target_plant: str = "Apple",
    batch_size: int = 64,
    num_workers: int = 0,
    image_size: int = 64,
):
    base_dataset, keep_indices, conditioned_dataset = build_filtered_dataset(
        dataset_root=dataset_root,
        target_plant=target_plant,
        image_size=image_size,
    )

    # Standard GAN sees only images.
    vanilla_subset = Subset(base_dataset, keep_indices)

    vanilla_loader = DataLoader(
        vanilla_subset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        drop_last=True,
    )

    # cGAN sees images + binary condition labels.
    conditional_loader = DataLoader(
        conditioned_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        drop_last=True,
    )

    return vanilla_loader, conditional_loader, base_dataset, keep_indices


def denorm(x: torch.Tensor) -> torch.Tensor:
    """[-1, 1] -> [0, 1]"""
    return (x * 0.5 + 0.5).clamp(0, 1)


def visualize_real_samples(loader: DataLoader, n_samples: int = 10, title: str = "Real Samples") -> None:
    """Visualize exactly n_samples images from loader."""
    if n_samples != 10:
        raise ValueError("This utility is designed to show exactly 10 images as requested.")

    images, _ = next(iter(loader))
    images = images[:n_samples]

    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    fig.suptitle(title, fontsize=14)
    for i, ax in enumerate(axes.flatten()):
        img = denorm(images[i]).permute(1, 2, 0).cpu().numpy()
        ax.imshow(img)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# ======================================
# Task 2 - Standard GAN Architecture
# ======================================


def weights_init(m: nn.Module) -> None:
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


class Generator(nn.Module):
    def __init__(self, z_dim: int = 100, img_channels: int = 3, feature_g: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, feature_g * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(feature_g * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(feature_g * 8, feature_g * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_g * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(feature_g * 4, feature_g * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_g * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(feature_g * 2, feature_g, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_g),
            nn.ReLU(True),
            nn.ConvTranspose2d(feature_g, img_channels, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class Discriminator(nn.Module):
    def __init__(self, img_channels: int = 3, feature_d: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(img_channels, feature_d, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_d, feature_d * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_d * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_d * 2, feature_d * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_d * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_d * 4, feature_d * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_d * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_d * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).view(-1)


def build_gan(
    z_dim: int = 100,
    img_channels: int = 3,
    feature_g: int = 64,
    feature_d: int = 64,
):
    gen = Generator(z_dim=z_dim, img_channels=img_channels, feature_g=feature_g).to(device)
    disc = Discriminator(img_channels=img_channels, feature_d=feature_d).to(device)
    gen.apply(weights_init)
    disc.apply(weights_init)
    return gen, disc

In [ ]:
# ==============================================
# Task 2/3 - GAN Training, Tracking, Comparison
# ==============================================


def save_generated_batch(
    generator: nn.Module,
    fixed_noise: torch.Tensor,
    epoch: int,
    out_dir: str,
    nrow: int = 8,
) -> str:
    os.makedirs(out_dir, exist_ok=True)
    generator.eval()
    with torch.no_grad():
        fake = generator(fixed_noise)
    generator.train()

    file_path = os.path.join(out_dir, f"epoch_{epoch:03d}.png")
    save_image(denorm(fake), file_path, nrow=nrow)
    return file_path


def train_gan(
    loader: DataLoader,
    epochs: int = 50,
    z_dim: int = 100,
    lr_disc: float = 4e-4,  # Required ratio setting.
    lr_gen: float = 1e-4,   # Required ratio setting.
    beta1: float = 0.5,
    beta2: float = 0.999,
    out_dir: str = "outputs/gan",
):
    gen, disc = build_gan(z_dim=z_dim)

    opt_disc = optim.Adam(disc.parameters(), lr=lr_disc, betas=(beta1, beta2))
    opt_gen = optim.Adam(gen.parameters(), lr=lr_gen, betas=(beta1, beta2))
    criterion = nn.BCELoss()

    fixed_noise = torch.randn(64, z_dim, 1, 1, device=device)

    history = {"d_loss": [], "g_loss": [], "saved_paths": {}}

    for epoch in range(1, epochs + 1):
        loop = tqdm(loader, desc=f"[GAN] Epoch {epoch}/{epochs}", leave=False)
        epoch_d_loss = 0.0
        epoch_g_loss = 0.0

        for real, _ in loop:
            real = real.to(device)
            batch_size = real.size(0)

            # ---------------------
            # Train Discriminator
            # ---------------------
            noise = torch.randn(batch_size, z_dim, 1, 1, device=device)
            fake = gen(noise)

            disc_real = disc(real)
            loss_disc_real = criterion(disc_real, torch.ones_like(disc_real))

            disc_fake = disc(fake.detach())
            loss_disc_fake = criterion(disc_fake, torch.zeros_like(disc_fake))

            loss_disc = (loss_disc_real + loss_disc_fake) / 2
            disc.zero_grad(set_to_none=True)
            loss_disc.backward()
            opt_disc.step()

            # -----------------
            # Train Generator
            # -----------------
            output = disc(fake)
            loss_gen = criterion(output, torch.ones_like(output))

            gen.zero_grad(set_to_none=True)
            loss_gen.backward()
            opt_gen.step()

            epoch_d_loss += loss_disc.item()
            epoch_g_loss += loss_gen.item()
            loop.set_postfix(d_loss=loss_disc.item(), g_loss=loss_gen.item())

        history["d_loss"].append(epoch_d_loss / len(loader))
        history["g_loss"].append(epoch_g_loss / len(loader))

        # Save snapshots every 5 epochs, and also epoch 1 for required comparison.
        if epoch == 1 or epoch % 5 == 0:
            path = save_generated_batch(gen, fixed_noise, epoch, out_dir)
            history["saved_paths"][epoch] = path

        # Epoch 10 checkpoint for mode-collapse inspection.
        if epoch == 10:
            print(
                "[Epoch 10 Check] Inspect generated images for diversity. "
                "If samples look nearly identical, mode collapse may be occurring."
            )

    return gen, disc, history


def compare_epochs_grid(history: Dict, epoch_a: int = 1, epoch_b: int = 50) -> None:
    if epoch_a not in history["saved_paths"] or epoch_b not in history["saved_paths"]:
        raise ValueError(
            f"Required snapshots not found. Available: {sorted(history['saved_paths'].keys())}"
        )

    img_a = np.array(Image.open(history["saved_paths"][epoch_a]))
    img_b = np.array(Image.open(history["saved_paths"][epoch_b]))

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(img_a)
    axes[0].set_title(f"Generated Samples - Epoch {epoch_a}")
    axes[0].axis("off")

    axes[1].imshow(img_b)
    axes[1].set_title(f"Generated Samples - Epoch {epoch_b}")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# ======================================
# Task 4 - Conditional GAN (cGAN)
# ======================================


class ConditionalGenerator(nn.Module):
    def __init__(
        self,
        z_dim: int = 100,
        num_classes: int = 2,
        emb_dim: int = 50,
        img_channels: int = 3,
        feature_g: int = 64,
    ):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, emb_dim)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim + emb_dim, feature_g * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(feature_g * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(feature_g * 8, feature_g * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_g * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(feature_g * 4, feature_g * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_g * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(feature_g * 2, feature_g, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_g),
            nn.ReLU(True),
            nn.ConvTranspose2d(feature_g, img_channels, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, noise: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        emb = self.label_emb(labels).unsqueeze(2).unsqueeze(3)
        x = torch.cat([noise, emb], dim=1)
        return self.net(x)


class ConditionalDiscriminator(nn.Module):
    def __init__(
        self,
        num_classes: int = 2,
        img_channels: int = 3,
        feature_d: int = 64,
        img_size: int = 64,
    ):
        super().__init__()
        self.img_size = img_size
        self.label_emb = nn.Embedding(num_classes, img_size * img_size)

        self.net = nn.Sequential(
            nn.Conv2d(img_channels + 1, feature_d, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_d, feature_d * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_d * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_d * 2, feature_d * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_d * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_d * 4, feature_d * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_d * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_d * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, images: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        emb = self.label_emb(labels).view(labels.size(0), 1, self.img_size, self.img_size)
        x = torch.cat([images, emb], dim=1)
        return self.net(x).view(-1)


def build_cgan(
    z_dim: int = 100,
    num_classes: int = 2,
    emb_dim: int = 50,
    img_size: int = 64,
):
    cgen = ConditionalGenerator(z_dim=z_dim, num_classes=num_classes, emb_dim=emb_dim).to(device)
    cdisc = ConditionalDiscriminator(num_classes=num_classes, img_size=img_size).to(device)
    cgen.apply(weights_init)
    cdisc.apply(weights_init)
    return cgen, cdisc

In [ ]:
def train_cgan(
    loader: DataLoader,
    epochs: int = 50,
    z_dim: int = 100,
    lr_disc: float = 4e-4,
    lr_gen: float = 1e-4,
    beta1: float = 0.5,
    beta2: float = 0.999,
    out_dir: str = "outputs/cgan",
    num_classes: int = 2,
):
    cgen, cdisc = build_cgan(z_dim=z_dim, num_classes=num_classes)

    opt_disc = optim.Adam(cdisc.parameters(), lr=lr_disc, betas=(beta1, beta2))
    opt_gen = optim.Adam(cgen.parameters(), lr=lr_gen, betas=(beta1, beta2))
    criterion = nn.BCELoss()

    fixed_noise = torch.randn(64, z_dim, 1, 1, device=device)
    fixed_labels = torch.tensor(([0] * 32) + ([1] * 32), device=device)

    history = {"d_loss": [], "g_loss": [], "saved_paths": {}}

    for epoch in range(1, epochs + 1):
        loop = tqdm(loader, desc=f"[cGAN] Epoch {epoch}/{epochs}", leave=False)
        epoch_d_loss = 0.0
        epoch_g_loss = 0.0

        for real, labels in loop:
            real = real.to(device)
            labels = labels.to(device)
            batch_size = real.size(0)

            # ---------------------
            # Train Discriminator
            # ---------------------
            noise = torch.randn(batch_size, z_dim, 1, 1, device=device)
            fake = cgen(noise, labels)

            disc_real = cdisc(real, labels)
            loss_disc_real = criterion(disc_real, torch.ones_like(disc_real))

            disc_fake = cdisc(fake.detach(), labels)
            loss_disc_fake = criterion(disc_fake, torch.zeros_like(disc_fake))

            loss_disc = (loss_disc_real + loss_disc_fake) / 2
            cdisc.zero_grad(set_to_none=True)
            loss_disc.backward()
            opt_disc.step()

            # -----------------
            # Train Generator
            # -----------------
            output = cdisc(fake, labels)
            loss_gen = criterion(output, torch.ones_like(output))

            cgen.zero_grad(set_to_none=True)
            loss_gen.backward()
            opt_gen.step()

            epoch_d_loss += loss_disc.item()
            epoch_g_loss += loss_gen.item()
            loop.set_postfix(d_loss=loss_disc.item(), g_loss=loss_gen.item())

        history["d_loss"].append(epoch_d_loss / len(loader))
        history["g_loss"].append(epoch_g_loss / len(loader))

        if epoch == 1 or epoch % 5 == 0:
            os.makedirs(out_dir, exist_ok=True)
            cgen.eval()
            with torch.no_grad():
                fake = cgen(fixed_noise, fixed_labels)
            cgen.train()

            save_path = os.path.join(out_dir, f"epoch_{epoch:03d}.png")
            save_image(denorm(fake), save_path, nrow=8)
            history["saved_paths"][epoch] = save_path

    return cgen, cdisc, history


def generate_leaf_type(
    cgen: nn.Module,
    leaf_type: str,
    z_dim: int = 100,
    n_samples: int = 8,
    out_path: str = "outputs/cgan/inference_request.png",
):
    """Inference utility: ask model for healthy or diseased leaves."""
    label_map = {"healthy": 0, "diseased": 1}
    key = leaf_type.lower().strip()
    if key not in label_map:
        raise ValueError("leaf_type must be 'healthy' or 'diseased'.")

    labels = torch.full((n_samples,), label_map[key], device=device, dtype=torch.long)
    noise = torch.randn(n_samples, z_dim, 1, 1, device=device)

    cgen.eval()
    with torch.no_grad():
        fake = cgen(noise, labels)
    cgen.train()

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    save_image(denorm(fake), out_path, nrow=min(4, n_samples))

    plt.figure(figsize=(8, 4))
    grid = make_grid(denorm(fake), nrow=min(4, n_samples)).permute(1, 2, 0).cpu().numpy()
    plt.imshow(grid)
    plt.title(f"Requested leaf type: {leaf_type.title()}")
    plt.axis("off")
    plt.show()

    print(f"Saved inference grid to: {out_path}")

## Run Configuration

Set your dataset location and choose one plant family (`"Apple"` or `"Tomato"`).

Expected structure for `ImageFolder`:

- PlantVillage/
  - Apple___healthy/
  - Apple___Black_rot/
  - ...
  - Tomato___healthy/
  - Tomato___Early_blight/
  - ...

In [ ]:
DATASET_ROOT = "./PlantVillage"  # Change to your local path
TARGET_PLANT = "Apple"            # "Apple" or "Tomato"
BATCH_SIZE = 64
EPOCHS = 50
Z_DIM = 100

vanilla_loader, conditional_loader, base_dataset, keep_indices = make_dataloaders(
    dataset_root=DATASET_ROOT,
    target_plant=TARGET_PLANT,
    batch_size=BATCH_SIZE,
    num_workers=0,
    image_size=64,
)

print(f"Total selected samples for {TARGET_PLANT}: {len(keep_indices)}")
print(f"Total original classes: {len(base_dataset.classes)}")

## Task 1 - Visualize Exactly 10 Real Samples

In [ ]:
visualize_real_samples(
    loader=vanilla_loader,
    n_samples=10,
    title=f"{TARGET_PLANT} Real Samples (Preprocessed 64x64)",
)

## Task 2 and Task 3 - Train Standard GAN, Save Snapshots, Compare Epoch 1 vs 50

In [ ]:
gan_gen, gan_disc, gan_history = train_gan(
    loader=vanilla_loader,
    epochs=EPOCHS,
    z_dim=Z_DIM,
    lr_disc=0.0004,  # Required setting
    lr_gen=0.0001,   # Required setting
    out_dir="outputs/gan",
)

compare_epochs_grid(gan_history, epoch_a=1, epoch_b=50)

## Task 4 - Train Conditional GAN and Generate Requested Leaf Type

In [ ]:
cgan_gen, cgan_disc, cgan_history = train_cgan(
    loader=conditional_loader,
    epochs=EPOCHS,
    z_dim=Z_DIM,
    lr_disc=0.0004,
    lr_gen=0.0001,
    out_dir="outputs/cgan",
    num_classes=2,
)

# User asks for specific condition
generate_leaf_type(cgan_gen, leaf_type="healthy", z_dim=Z_DIM, n_samples=8)
generate_leaf_type(cgan_gen, leaf_type="diseased", z_dim=Z_DIM, n_samples=8,
                   out_path="outputs/cgan/inference_request_diseased.png")